
# ARC-v0.17 — Deployable Risk → Selective Fidelity Closure

This notebook closes the remaining end-to-end gap in the paper:

\[
\text{deployable PQ32-only risk}
\rightarrow
\text{selective SQ8 feedback}
\rightarrow
\text{held-out FEVER utility}
\rightarrow
\text{matched-budget random baseline}
\rightarrow
\text{measured incremental cost}.
\]

## Primary question

On the untouched FEVER validation split, does a **fit-frozen, deployment-feasible PQ32-only risk score** improve retrieval utility when it is used to decide which queries receive SQ8 feedback?

## Policies

All policies keep **PQ32 search/evaluation** fixed. Only the feedback source changes.

1. **Always-PQ32** — PQ32 search → PQ32 feedback.
2. **Always-SQ8-feedback** — PQ32 search → SQ8 feedback.
3. **Random-selective SQ8** — budget-matched random allocation of SQ8 feedback.
4. **Deployable-risk-selective SQ8** — fit-frozen ARC-v0.15/v0.16.1 PQ32-only risk model allocates SQ8 feedback.

Primary budget:

\[
B=25\%.
\]

Secondary sensitivity:

\[
B\in\{10\%,25\%,50\%\}.
\]

## Statistical unit

The primary statistical unit is the **query**. Utility contrasts use paired query-level bootstrap intervals.

## Claim discipline

This is a post-confirmatory closure experiment. It does **not** change the sealed H1–H4 confirmation. It tests whether the existing deployable risk score is actionable in the same FEVER setting in which it was validated.


In [ ]:

# ============================================================
# Cell 1 — Install / imports / Drive
# ============================================================

%pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn pandas numpy

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import inspect
import json
import math
import os
import random
import time
import warnings
import gc

import numpy as np
import pandas as pd
import psutil
import faiss

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260816
rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)

BUDGETS = [0.10, 0.25, 0.50]
PRIMARY_BUDGET = 0.25
BOOTSTRAP_REPS = 2000

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed."

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
assert ARC_ROOT.is_dir(), ARC_ROOT

print("Drive:", DRIVE_ROOT)
print("ARC root:", ARC_ROOT)


In [ ]:

# ============================================================
# Cell 2 — Locate frozen FEVER sources and verify integrity
# ============================================================

V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"
assert V013_ROOT.is_dir(), V013_ROOT

PREFERRED_RUN = V013_ROOT / "20260817-140640"

def is_complete_v013(path):
    return (
        path.is_dir()
        and len(list(path.glob("fit-*.parquet"))) == 44
        and len(list(path.glob("validation-*.parquet"))) == 44
        and (path / "v013_fever_boundary_protocol.json").is_file()
        and (path / "v013_validation_continuation_report.json").is_file()
    )

if is_complete_v013(PREFERRED_RUN):
    V013_RUN = PREFERRED_RUN
else:
    candidates = sorted(
        [p for p in V013_ROOT.iterdir() if p.is_dir() and is_complete_v013(p)],
        reverse=True,
    )
    assert candidates, "No complete ARC-v0.13 run found."
    V013_RUN = candidates[0]

PROTOCOL_PATH = V013_RUN / "v013_fever_boundary_protocol.json"
VALIDATION_REPORT_PATH = V013_RUN / "v013_validation_continuation_report.json"

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))

assert protocol["status"] == "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP"
assert validation_report["status"] == "ARC_V013_FEVER_VALIDATION_CONTINUATION_COMPLETE"
assert protocol["test_access_allowed"] is False
assert validation_report["test_accessed"] is False

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

print("v0.13 source:", V013_RUN)
print("protocol SHA:", sha256_file(PROTOCOL_PATH))
print("validation report SHA:", sha256_file(VALIDATION_REPORT_PATH))
print("FROZEN V0.13 SOURCE — PASS")


In [ ]:

# ============================================================
# Cell 3 — FEVER corpus/query/index paths
# ============================================================

DIM = 384
N_DOCS = 5_416_568
NPROBE = 64
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

ROOT = (
    DRIVE_ROOT
    / "hc-rars-fever-5m-untouched-confirmation-v1"
)

INDEX_ROOT = (
    DRIVE_ROOT
    / "rag-pq-checkpoints"
    / "arc-index-cache"
)

CORPUS_MEMMAP = ROOT / "stage1/corpus_embeddings.float16.memmap"
QUERY_EMB = ROOT / "stage1/query_embeddings_v2.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
DEV_QRELS = ROOT / "stage2/dev_qrels_rows.csv"

PQ32_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
)

SQ8_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss"
)

required = {
    "CORPUS_MEMMAP": CORPUS_MEMMAP,
    "QUERY_EMB": QUERY_EMB,
    "QUERY_IDS": QUERY_IDS,
    "SPLIT_MANIFEST": SPLIT_MANIFEST,
    "DEV_QRELS": DEV_QRELS,
    "PQ32_INDEX": PQ32_PATH,
    "SQ8_INDEX": SQ8_PATH,
}

for name, path in required.items():
    print(f"{name:20s}", "OK" if path.is_file() else "MISSING", path)
    assert path.is_file(), path

print("FEVER DATA PREFLIGHT — PASS")


In [ ]:

# ============================================================
# Cell 4 — Load corpus/query data and qrels safely
# ============================================================

corpus = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(N_DOCS, DIM),
)

queries = np.load(
    QUERY_EMB,
    mmap_mode="r",
)

query_ids = [
    line.strip()
    for line in QUERY_IDS.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert queries.shape == (6666, DIM)
assert len(query_ids) == 6666

query_id_to_row = {
    str(qid): i
    for i, qid in enumerate(query_ids)
}

split_manifest = json.loads(
    SPLIT_MANIFEST.read_text(encoding="utf-8")
)

qrels_raw = pd.read_csv(DEV_QRELS)

print("Qrels columns:", qrels_raw.columns.tolist())

# FEVER corpus-id can be string-valued (e.g., Wikipedia titles),
# while corpus-row is the integer row index used by the FAISS/memmap artifacts.
qid_col = "query-id"
row_col = "corpus-row"
rel_col = "score"

assert {qid_col, row_col, rel_col}.issubset(qrels_raw.columns)

qrels_by_query = {}

for qid, g in qrels_raw.groupby(qid_col):
    rel_rows = set(
        int(row)
        for row, rel in zip(g[row_col], g[rel_col])
        if float(rel) > 0
    )
    qrels_by_query[str(qid)] = rel_rows

print("Queries with qrels:", len(qrels_by_query))
print("DATA LOAD — PASS")


In [ ]:

# ============================================================
# Cell 5 — Deterministic FEVER FIT / validation split
# ============================================================

import hashlib as _hashlib

def split_is_fit(qid):
    h = _hashlib.sha256(str(qid).encode("utf-8")).digest()
    return (h[0] % 2) == 0

fit_query_ids = [
    qid
    for qid in query_ids
    if split_is_fit(qid)
]

val_query_ids = [
    qid
    for qid in query_ids
    if not split_is_fit(qid)
]

assert len(fit_query_ids) == 3350, len(fit_query_ids)
assert len(val_query_ids) == 3316, len(val_query_ids)

fit_rows = np.array(
    [query_id_to_row[qid] for qid in fit_query_ids],
    dtype=np.int64,
)

val_rows = np.array(
    [query_id_to_row[qid] for qid in val_query_ids],
    dtype=np.int64,
)

print("FIT queries:", len(fit_rows))
print("VAL queries:", len(val_rows))
print("SPLIT RECONSTRUCTION — PASS")


In [ ]:

# ============================================================
# Cell 6 — Load FAISS indexes and low-level utilities
# ============================================================

pq32 = faiss.read_index(str(PQ32_PATH))
sq8 = faiss.read_index(str(SQ8_PATH))

for index in [pq32, sq8]:
    if hasattr(index, "nprobe"):
        index.nprobe = NPROBE

def l2_normalize_rows(x):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n = np.maximum(n, 1e-12)
    return x / n

def l2_normalize_vec(x):
    x = np.asarray(x, dtype=np.float32)
    n = float(np.linalg.norm(x))
    if n <= 1e-12:
        return x
    return x / n

def fetch_doc_vectors(rows):
    rows = np.asarray(rows, dtype=np.int64)
    return np.asarray(corpus[rows], dtype=np.float32)

def ndcg_at_10(retrieved_rows, relevant_rows):
    rel = np.array(
        [1.0 if int(r) in relevant_rows else 0.0 for r in retrieved_rows[:10]],
        dtype=np.float64,
    )

    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))
    dcg = float(np.sum(rel * discounts))

    ideal_len = min(10, len(relevant_rows))
    if ideal_len == 0:
        return 0.0

    ideal_rel = np.ones(ideal_len, dtype=np.float64)
    idcg = float(
        np.sum(
            ideal_rel
            * (1.0 / np.log2(np.arange(2, ideal_len + 2)))
        )
    )

    return dcg / idcg if idcg > 0 else 0.0

def retrieve(index, q, topn=TOP_RETRIEVE):
    q = l2_normalize_vec(q).reshape(1, -1).astype(np.float32)
    scores, ids = index.search(q, topn)
    return scores[0], ids[0]

print("PQ32 ntotal:", pq32.ntotal)
print("SQ8 ntotal :", sq8.ntotal)
print("INDEX LOAD — PASS")


In [ ]:

# ============================================================
# Cell 7 — Feedback and hybrid trajectory simulator
# ============================================================

def feedback_vector(index, q, cfg):
    scores, ids = retrieve(index, q, TOP_RETRIEVE)

    k = int(cfg["k"])
    top_ids = ids[:k]
    top_scores = scores[:k]

    docs = fetch_doc_vectors(top_ids)
    docs = l2_normalize_rows(docs)

    if cfg["method"] == "mean":
        fb = docs.mean(axis=0)

    elif cfg["method"] == "softmax":
        temp = float(cfg["temperature"])
        z = top_scores.astype(np.float64) / temp
        z = z - np.max(z)
        w = np.exp(z)
        w = w / np.sum(w)
        fb = np.sum(
            docs * w[:, None].astype(np.float32),
            axis=0,
        )

    else:
        raise ValueError(cfg["method"])

    return l2_normalize_vec(fb)

def next_query(q0, fb, alpha):
    return l2_normalize_vec(
        (1.0 - float(alpha)) * q0
        + float(alpha) * fb
    )

def run_feedback_policy_one_query(
    query_row,
    cfg,
    feedback_source_by_round,
):
    '''
    Search/evaluation is always PQ32.

    feedback_source_by_round:
      list length MAX_ROUNDS with values "PQ32" or "SQ8".
    '''

    q0 = l2_normalize_vec(
        np.asarray(
            queries[query_row],
            dtype=np.float32,
        )
    )

    q = q0.copy()

    qid = query_ids[query_row]
    relevant = qrels_by_query.get(
        str(qid),
        set(),
    )

    utilities = []

    # round 0 + MAX_ROUNDS updated states
    for t in range(MAX_ROUNDS + 1):
        pq_scores, pq_ids = retrieve(
            pq32,
            q,
            TOP_RETRIEVE,
        )

        utilities.append(
            ndcg_at_10(
                pq_ids,
                relevant,
            )
        )

        if t == MAX_ROUNDS:
            break

        source = feedback_source_by_round[t]

        if source == "PQ32":
            fb = feedback_vector(
                pq32,
                q,
                cfg,
            )
        elif source == "SQ8":
            fb = feedback_vector(
                sq8,
                q,
                cfg,
            )
        else:
            raise ValueError(source)

        q = next_query(
            q0,
            fb,
            cfg["alpha"],
        )

    return {
        "query_id": str(qid),
        "query_row": int(query_row),
        "u0": float(utilities[0]),
        "u1": float(utilities[1]),
        "u2": float(utilities[2]),
        "u3": float(utilities[3]),
        "u4": float(utilities[4]),
        "final_ndcg10": float(utilities[-1]),
        "mean_ndcg10": float(np.mean(utilities)),
    }

print("HYBRID TRAJECTORY SIMULATOR — READY")



## Policy configuration used for closure

To avoid opening a new hyperparameter search, ARC-v0.17 uses the same two frozen feedback configurations already used for the HotpotQA selective-fidelity analysis:

- **mean**: \(k=20,\alpha=0.3\)
- **softmax**: \(k=5,\alpha=0.5,\tau=0.1\)

The primary closure claim is evaluated separately for each configuration.


In [ ]:

# ============================================================
# Cell 8 — Frozen policy configurations
# ============================================================

POLICY_CONFIGS = [
    {
        "name": "mean-k20-a0p3",
        "method": "mean",
        "alpha": 0.3,
        "k": 20,
        "temperature": None,
    },
    {
        "name": "softmax-k5-a0p5-t0p1",
        "method": "softmax",
        "alpha": 0.5,
        "k": 5,
        "temperature": 0.1,
    },
]

print(json.dumps(POLICY_CONFIGS, indent=2))


In [ ]:

# ============================================================
# Cell 9 — Recover exact PQ32-only initial-query features
# ============================================================

required_feature_cols = {
    "query_id",
    "pq32_entropy20",
    "pq32_margin1_10",
}

feature_matches = []

for p in V013_ROOT.rglob("*.parquet"):
    try:
        df = pd.read_parquet(p)
    except Exception:
        continue

    if required_feature_cols.issubset(df.columns):
        tmp = (
            df[
                [
                    "query_id",
                    "pq32_entropy20",
                    "pq32_margin1_10",
                ]
            ]
            .copy()
        )
        tmp["query_id"] = tmp["query_id"].astype(str)

        if tmp["query_id"].nunique() >= 6666:
            feature_matches.append(
                (
                    p,
                    tmp.drop_duplicates("query_id"),
                )
            )

assert feature_matches, (
    "No full-coverage PQ32 query-feature artifact found."
)

# Prefer artifact physically inside the frozen v0.13 run if present.
frozen_matches = [
    item
    for item in feature_matches
    if str(item[0]).startswith(str(V013_RUN))
]

if frozen_matches:
    FEATURE_SOURCE, baseline_features = frozen_matches[0]
else:
    FEATURE_SOURCE, baseline_features = feature_matches[0]

baseline_features = baseline_features[
    [
        "query_id",
        "pq32_entropy20",
        "pq32_margin1_10",
    ]
].copy()

assert baseline_features["query_id"].nunique() == 6666

print("Feature source:", FEATURE_SOURCE)
print("Feature SHA:", sha256_file(FEATURE_SOURCE))
print("Feature queries:", baseline_features["query_id"].nunique())
print("FEATURE RECOVERY — PASS")


In [ ]:

# ============================================================
# Cell 10 — Reconstruct FIT H3 target to fit deployable predictor
# ============================================================

fit_files = sorted(V013_RUN.glob("fit-*.parquet"))
assert len(fit_files) == 44

fit_traj = pd.concat(
    [pd.read_parquet(p) for p in fit_files],
    ignore_index=True,
)

GROUP_COLS = [
    "query_id",
    "low",
    "high",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

rows = []

for keys, g in fit_traj.groupby(
    GROUP_COLS,
    dropna=False,
    sort=False,
):
    g = g.sort_values("iteration")
    x = g["iteration"].to_numpy(np.float64)
    y = g["abs_utility_gap"].to_numpy(np.float64)

    row = dict(zip(GROUP_COLS, keys))
    row["H3_abs_slope"] = float(
        np.polyfit(x, y, 1)[0]
    )
    rows.append(row)

fit_slopes = pd.DataFrame(rows)

fit_slopes["is_amplifying_abs"] = (
    fit_slopes["H3_abs_slope"] > 0.002
).astype(int)

fit_model = fit_slopes.merge(
    baseline_features,
    on="query_id",
    how="left",
    validate="many_to_one",
)

fit_model["is_softmax"] = (
    fit_model["method"] == "softmax"
).astype(float)

fit_model["temperature_numeric"] = (
    pd.to_numeric(
        fit_model["temperature"],
        errors="coerce",
    )
    .fillna(1.0)
)

fit_model["log_k"] = np.log(
    fit_model["k"].astype(float)
)

DEPLOYABLE_FEATURES = [
    "pq32_entropy20",
    "pq32_margin1_10",
    "alpha",
    "log_k",
    "is_softmax",
    "temperature_numeric",
]

clf = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

clf.fit(
    fit_model[DEPLOYABLE_FEATURES].to_numpy(np.float64),
    fit_model["is_amplifying_abs"].to_numpy(int),
)

print("Deployable predictor fit rows:", len(fit_model))
print("DEPLOYABLE PREDICTOR FIT — PASS")


In [ ]:

# ============================================================
# Cell 11 — Freeze validation risk ranking for each policy config
# ============================================================

val_feature_table = (
    pd.DataFrame({
        "query_id": [str(q) for q in val_query_ids],
    })
    .merge(
        baseline_features,
        on="query_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(val_feature_table) == 3316
assert val_feature_table[
    [
        "pq32_entropy20",
        "pq32_margin1_10",
    ]
].notna().all().all()

risk_tables = {}

for cfg in POLICY_CONFIGS:
    t = val_feature_table.copy()

    t["alpha"] = float(cfg["alpha"])
    t["log_k"] = np.log(float(cfg["k"]))
    t["is_softmax"] = float(cfg["method"] == "softmax")
    t["temperature_numeric"] = (
        1.0
        if cfg["temperature"] is None
        else float(cfg["temperature"])
    )

    t["risk_score"] = clf.predict_proba(
        t[DEPLOYABLE_FEATURES].to_numpy(np.float64)
    )[:, 1]

    t = t.sort_values(
        "risk_score",
        ascending=False,
    ).reset_index(drop=True)

    risk_tables[cfg["name"]] = t

    print(
        cfg["name"],
        "risk range:",
        float(t["risk_score"].min()),
        float(t["risk_score"].max()),
    )

print("VALIDATION RISK RANKINGS — FROZEN")


In [ ]:

# ============================================================
# Cell 12 — Create output directory and checkpoint helpers
# ============================================================

OUT_ROOT = ARC_ROOT / "fever-deployable-selective-fidelity-closure-v017"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

def checkpoint_path(cfg_name, policy_name, budget=None):
    suffix = (
        policy_name
        if budget is None
        else f"{policy_name}-b{int(round(100*budget)):02d}"
    )
    return OUT / f"{cfg_name}__{suffix}.parquet"

print("Output:", OUT)


In [ ]:

# ============================================================
# Cell 13 — Policy runner with resumable checkpoints
# ============================================================

def run_policy_checkpoint(
    cfg,
    policy_name,
    selected_sq8_query_ids,
    checkpoint_file,
):
    if checkpoint_file.is_file():
        df = pd.read_parquet(checkpoint_file)
        print("REUSE:", checkpoint_file.name, df.shape)
        return df

    selected_sq8_query_ids = set(
        str(x)
        for x in selected_sq8_query_ids
    )

    rows = []

    t0 = time.perf_counter()

    for i, query_row in enumerate(val_rows, 1):
        qid = str(query_ids[int(query_row)])

        use_sq8 = qid in selected_sq8_query_ids

        if policy_name == "always_pq32":
            sources = ["PQ32"] * MAX_ROUNDS

        elif policy_name == "always_sq8_feedback":
            sources = ["SQ8"] * MAX_ROUNDS

        elif policy_name in {
            "risk_selective",
            "random_selective",
        }:
            sources = (
                ["SQ8"] * MAX_ROUNDS
                if use_sq8
                else ["PQ32"] * MAX_ROUNDS
            )

        else:
            raise ValueError(policy_name)

        result = run_feedback_policy_one_query(
            int(query_row),
            cfg,
            sources,
        )

        result["policy"] = policy_name
        result["sq8_selected"] = bool(use_sq8)
        result["config_name"] = cfg["name"]

        rows.append(result)

        if i in [1, 250, 500, 1000, 1500, 2000, 2500, 3000, len(val_rows)]:
            print(
                f"[{i:04d}/{len(val_rows)}]",
                cfg["name"],
                policy_name,
                "elapsed=",
                time.perf_counter() - t0,
            )

    df = pd.DataFrame(rows)

    df.to_parquet(
        checkpoint_file,
        index=False,
    )

    print(
        "SAVED:",
        checkpoint_file.name,
        "rows=",
        len(df),
        "seconds=",
        time.perf_counter() - t0,
    )

    return df



## Execution order

The next cell runs the two fixed configurations. For each configuration:

- Always-PQ32 is run once.
- Always-SQ8-feedback is run once.
- Risk-selective is run at 10/25/50%.
- Random-selective uses a deterministic, **budget-matched** random query subset for each budget.

All checkpoints are persisted immediately to Drive and are resumable.


In [ ]:

# ============================================================
# Cell 14 — Run FEVER closure experiment
# ============================================================

all_results = []

for cfg in POLICY_CONFIGS:
    cfg_name = cfg["name"]

    print("\n" + "=" * 90)
    print("CONFIG:", cfg_name)
    print("=" * 90)

    risk = risk_tables[cfg_name]

    # Always-PQ32
    df = run_policy_checkpoint(
        cfg,
        "always_pq32",
        [],
        checkpoint_path(
            cfg_name,
            "always_pq32",
        ),
    )
    all_results.append(df)

    # Always-SQ8 feedback
    df = run_policy_checkpoint(
        cfg,
        "always_sq8_feedback",
        val_query_ids,
        checkpoint_path(
            cfg_name,
            "always_sq8_feedback",
        ),
    )
    all_results.append(df)

    for budget in BUDGETS:
        n_select = int(
            round(
                budget * len(val_query_ids)
            )
        )

        risk_selected = set(
            risk.iloc[:n_select]["query_id"].astype(str)
        )

        # deterministic matched random allocation
        budget_seed = (
            SEED
            + int(round(100 * budget))
            + sum(ord(c) for c in cfg_name)
        )

        local_rng = np.random.default_rng(
            budget_seed
        )

        random_selected = set(
            local_rng.choice(
                np.array(val_query_ids, dtype=object),
                size=n_select,
                replace=False,
            )
        )

        risk_df = run_policy_checkpoint(
            cfg,
            "risk_selective",
            risk_selected,
            checkpoint_path(
                cfg_name,
                "risk_selective",
                budget,
            ),
        )
        risk_df["budget"] = budget
        all_results.append(risk_df)

        random_df = run_policy_checkpoint(
            cfg,
            "random_selective",
            random_selected,
            checkpoint_path(
                cfg_name,
                "random_selective",
                budget,
            ),
        )
        random_df["budget"] = budget
        all_results.append(random_df)

print("\nARC-v0.17 POLICY EXECUTION — COMPLETE")


In [ ]:

# ============================================================
# Cell 15 — Reload canonical checkpoint set and summarize
# ============================================================

records = []

for cfg in POLICY_CONFIGS:
    cfg_name = cfg["name"]

    for policy_name in [
        "always_pq32",
        "always_sq8_feedback",
    ]:
        p = checkpoint_path(
            cfg_name,
            policy_name,
        )
        assert p.is_file(), p

        df = pd.read_parquet(p)
        df["budget"] = np.nan
        records.append(df)

    for budget in BUDGETS:
        for policy_name in [
            "risk_selective",
            "random_selective",
        ]:
            p = checkpoint_path(
                cfg_name,
                policy_name,
                budget,
            )
            assert p.is_file(), p

            df = pd.read_parquet(p)
            df["budget"] = budget
            records.append(df)

results = pd.concat(
    records,
    ignore_index=True,
)

summary = (
    results
    .groupby(
        [
            "config_name",
            "policy",
            "budget",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        queries=("query_id", "nunique"),
        final_ndcg10=("final_ndcg10", "mean"),
        mean_ndcg10=("mean_ndcg10", "mean"),
        sq8_selected_fraction=("sq8_selected", "mean"),
    )
)

display(summary)

summary.to_csv(
    OUT / "v017_policy_quality_summary.csv",
    index=False,
)

print("QUALITY SUMMARY — COMPLETE")


In [ ]:

# ============================================================
# Cell 16 — Paired query-level bootstrap contrasts
# ============================================================

def paired_bootstrap_delta(
    a,
    b,
    value_col="final_ndcg10",
    reps=BOOTSTRAP_REPS,
    seed=SEED,
):
    m = (
        a[["query_id", value_col]]
        .rename(columns={value_col: "a"})
        .merge(
            b[["query_id", value_col]]
            .rename(columns={value_col: "b"}),
            on="query_id",
            how="inner",
            validate="one_to_one",
        )
    )

    assert len(m) == 3316

    diff = (
        m["a"].to_numpy(np.float64)
        - m["b"].to_numpy(np.float64)
    )

    point = float(diff.mean())

    rng = np.random.default_rng(seed)
    n = len(diff)

    boots = np.empty(reps, dtype=np.float64)

    for i in range(reps):
        idx = rng.integers(
            0,
            n,
            size=n,
        )
        boots[i] = float(
            diff[idx].mean()
        )

    return {
        "estimate": point,
        "ci_low": float(np.quantile(boots, 0.025)),
        "ci_high": float(np.quantile(boots, 0.975)),
        "p_le_0": float((boots <= 0).mean()),
    }

contrast_rows = []

for cfg in POLICY_CONFIGS:
    cfg_name = cfg["name"]

    base = results[
        (results["config_name"] == cfg_name)
        & (results["policy"] == "always_pq32")
    ]

    upper = results[
        (results["config_name"] == cfg_name)
        & (results["policy"] == "always_sq8_feedback")
    ]

    for budget in BUDGETS:
        risk_df = results[
            (results["config_name"] == cfg_name)
            & (results["policy"] == "risk_selective")
            & np.isclose(results["budget"], budget, equal_nan=False)
        ]

        rand_df = results[
            (results["config_name"] == cfg_name)
            & (results["policy"] == "random_selective")
            & np.isclose(results["budget"], budget, equal_nan=False)
        ]

        for label, a, b in [
            ("risk_vs_pq32", risk_df, base),
            ("risk_vs_random", risk_df, rand_df),
            ("random_vs_pq32", rand_df, base),
            ("always_sq8_vs_pq32", upper, base),
        ]:
            stat = paired_bootstrap_delta(
                a,
                b,
                seed=(
                    SEED
                    + int(100 * budget)
                    + sum(ord(c) for c in cfg_name + label)
                ),
            )

            contrast_rows.append({
                "config_name": cfg_name,
                "budget": budget,
                "contrast": label,
                **stat,
            })

contrasts = pd.DataFrame(
    contrast_rows
)

display(contrasts)

contrasts.to_csv(
    OUT / "v017_paired_quality_contrasts.csv",
    index=False,
)

print("PAIRED QUERY BOOTSTRAP — COMPLETE")


In [ ]:

# ============================================================
# Cell 17 — Benefit recovery and primary decision
# ============================================================

decision_rows = []

for cfg in POLICY_CONFIGS:
    cfg_name = cfg["name"]

    s = summary[
        summary["config_name"] == cfg_name
    ]

    base_q = float(
        s.loc[
            s["policy"] == "always_pq32",
            "final_ndcg10",
        ].iloc[0]
    )

    upper_q = float(
        s.loc[
            s["policy"] == "always_sq8_feedback",
            "final_ndcg10",
        ].iloc[0]
    )

    full_benefit = upper_q - base_q

    for budget in BUDGETS:
        risk_q = float(
            s.loc[
                (s["policy"] == "risk_selective")
                & np.isclose(s["budget"], budget, equal_nan=False),
                "final_ndcg10",
            ].iloc[0]
        )

        rand_q = float(
            s.loc[
                (s["policy"] == "random_selective")
                & np.isclose(s["budget"], budget, equal_nan=False),
                "final_ndcg10",
            ].iloc[0]
        )

        recovery = (
            (risk_q - base_q) / full_benefit
            if abs(full_benefit) > 1e-12
            else np.nan
        )

        primary_contrast = contrasts[
            (contrasts["config_name"] == cfg_name)
            & np.isclose(contrasts["budget"], budget)
            & (contrasts["contrast"] == "risk_vs_random")
        ].iloc[0]

        decision_rows.append({
            "config_name": cfg_name,
            "budget": budget,
            "always_pq32_final_ndcg10": base_q,
            "always_sq8_feedback_final_ndcg10": upper_q,
            "risk_selective_final_ndcg10": risk_q,
            "random_selective_final_ndcg10": rand_q,
            "always_sq8_benefit": full_benefit,
            "risk_recovery_fraction": recovery,
            "risk_minus_random": float(primary_contrast["estimate"]),
            "risk_minus_random_ci_low": float(primary_contrast["ci_low"]),
            "risk_minus_random_ci_high": float(primary_contrast["ci_high"]),
            "risk_minus_random_p_le_0": float(primary_contrast["p_le_0"]),
        })

decision_table = pd.DataFrame(
    decision_rows
)

display(decision_table)

decision_table.to_csv(
    OUT / "v017_closure_decision_table.csv",
    index=False,
)

print("\nPRIMARY 25% DECISION")
print("--------------------")

primary = decision_table[
    np.isclose(
        decision_table["budget"],
        PRIMARY_BUDGET,
    )
]

display(primary)

for _, row in primary.iterrows():
    cfg_name = row["config_name"]

    if (
        row["risk_minus_random"] > 0
        and row["risk_minus_random_ci_low"] > 0
    ):
        print(
            cfg_name,
            "PASS: deployable risk selector beats matched random at 25%."
        )
    elif row["risk_minus_random"] > 0:
        print(
            cfg_name,
            "BORDERLINE/NULL: point estimate positive but CI crosses zero."
        )
    else:
        print(
            cfg_name,
            "FAIL/NEGATIVE: deployable selector does not beat random."
        )


In [ ]:

# ============================================================
# Cell 18 — Local measured runtime audit
#
# Measure the actual hybrid execution cost for a small fixed
# validation subset, with the selection masks already frozen.
# This is not a production benchmark.
# ============================================================

RUNTIME_QUERY_COUNT = min(
    500,
    len(val_rows),
)

runtime_rows_subset = val_rows[:RUNTIME_QUERY_COUNT]

runtime_records = []

for cfg in POLICY_CONFIGS:
    cfg_name = cfg["name"]
    risk = risk_tables[cfg_name]

    for budget in [PRIMARY_BUDGET]:
        n_select = int(
            round(
                budget * len(val_query_ids)
            )
        )

        risk_selected = set(
            risk.iloc[:n_select]["query_id"].astype(str)
        )

        budget_seed = (
            SEED
            + int(round(100 * budget))
            + sum(ord(c) for c in cfg_name)
        )

        local_rng = np.random.default_rng(
            budget_seed
        )

        random_selected = set(
            local_rng.choice(
                np.array(val_query_ids, dtype=object),
                size=n_select,
                replace=False,
            )
        )

        policies = {
            "always_pq32": set(),
            "always_sq8_feedback": set(val_query_ids),
            "risk_selective": risk_selected,
            "random_selective": random_selected,
        }

        for policy_name, selected in policies.items():
            t0 = time.perf_counter()

            for query_row in runtime_rows_subset:
                qid = str(query_ids[int(query_row)])
                use_sq8 = qid in selected

                if policy_name == "always_pq32":
                    sources = ["PQ32"] * MAX_ROUNDS
                elif policy_name == "always_sq8_feedback":
                    sources = ["SQ8"] * MAX_ROUNDS
                else:
                    sources = (
                        ["SQ8"] * MAX_ROUNDS
                        if use_sq8
                        else ["PQ32"] * MAX_ROUNDS
                    )

                _ = run_feedback_policy_one_query(
                    int(query_row),
                    cfg,
                    sources,
                )

            seconds = time.perf_counter() - t0

            runtime_records.append({
                "config_name": cfg_name,
                "budget": budget,
                "policy": policy_name,
                "query_count": RUNTIME_QUERY_COUNT,
                "seconds": seconds,
                "ms_per_query": (
                    1000.0 * seconds / RUNTIME_QUERY_COUNT
                ),
            })

runtime_df = pd.DataFrame(
    runtime_records
)

display(runtime_df)

runtime_df.to_csv(
    OUT / "v017_local_runtime_audit.csv",
    index=False,
)

print("LOCAL RUNTIME AUDIT — COMPLETE")


In [ ]:

# ============================================================
# Cell 19 — Final report / claim gate
# ============================================================

primary_rows = decision_table[
    np.isclose(
        decision_table["budget"],
        PRIMARY_BUDGET,
    )
].copy()

all_primary_pass = bool(
    (
        (primary_rows["risk_minus_random"] > 0)
        & (primary_rows["risk_minus_random_ci_low"] > 0)
    ).all()
)

any_primary_pass = bool(
    (
        (primary_rows["risk_minus_random"] > 0)
        & (primary_rows["risk_minus_random_ci_low"] > 0)
    ).any()
)

if all_primary_pass:
    claim_gate = "STRONG_CLOSURE"
elif any_primary_pass:
    claim_gate = "PARTIAL_CLOSURE"
else:
    claim_gate = "NO_RELIABLE_CLOSURE"

report = {
    "status":
        "ARC_V017_DEPLOYABLE_SELECTIVE_FIDELITY_CLOSURE_COMPLETE",

    "source_v013_run":
        str(V013_RUN),

    "source_v013_protocol_sha256":
        sha256_file(PROTOCOL_PATH),

    "source_v013_validation_report_sha256":
        sha256_file(VALIDATION_REPORT_PATH),

    "feature_source":
        str(FEATURE_SOURCE),

    "feature_source_sha256":
        sha256_file(FEATURE_SOURCE),

    "primary_budget":
        PRIMARY_BUDGET,

    "secondary_budgets":
        BUDGETS,

    "policy_configs":
        POLICY_CONFIGS,

    "claim_gate":
        claim_gate,

    "all_primary_configs_pass":
        all_primary_pass,

    "any_primary_config_pass":
        any_primary_pass,

    "test_accessed":
        False,

    "interpretation_constraints": [
        (
            "Search/evaluation stays on PQ32; only the feedback source "
            "is selectively upgraded to SQ8."
        ),
        (
            "The deployable selector is fit only on FEVER FIT and "
            "applied without retuning to untouched FEVER validation."
        ),
        (
            "Runtime measurements are local single-machine measurements, "
            "not production throughput or latency guarantees."
        ),
        (
            "This is a post-confirmatory closure experiment and does not "
            "alter the sealed H1-H4 confirmation."
        ),
    ],

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = (
    OUT
    / "v017_deployable_selective_fidelity_closure_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(
    REPORT_PATH
)

(
    OUT
    / "V017_REPORT_SHA256.txt"
).write_text(
    report_sha
    + "  "
    + REPORT_PATH.name
    + "\n",
    encoding="utf-8",
)

print()
print("=" * 90)
print("ARC-v0.17 DEPLOYABLE SELECTIVE-FIDELITY CLOSURE — COMPLETE")
print("=" * 90)
print("Claim gate:", claim_gate)
print("Output:", OUT)
print("Report SHA-256:", report_sha)
print("Test accessed:", False)
print("=" * 90)
